In [4]:
# 1. Imports
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import pandas as pd
import matplotlib.pyplot as plt


In [ ]:
# 2. Chargement des données brutes (depuis le json)
bronze_path = "../data/water_quality_2026.json"
spark = SparkSession.builder.appName("RawJSON").getOrCreate()
df_bronze = spark.read.json(bronze_path)


In [10]:
# 3. Exploration initiale
print("=== STRUCTURE DES DONNÉES ===")
df_bronze.printSchema()

print(f"\n=== NOMBRE DE LIGNES: {df_bronze.count()} ===")

print("\n=== PREMIÈRES LIGNES ===")
df_bronze.limit(5).toPandas()

print("\n=== STATISTIQUES NUMÉRIQUES ===")
df_bronze.select([c for c in df_bronze.columns if df_bronze.schema[c].dataType.typeName() in ['int', 'double']]).describe().show()


=== STRUCTURE DES DONNÉES ===
root
 |-- code_commune: string (nullable = true)
 |-- code_departement: string (nullable = true)
 |-- code_installation_amont: string (nullable = true)
 |-- code_lieu_analyse: string (nullable = true)
 |-- code_parametre: string (nullable = true)
 |-- code_parametre_cas: string (nullable = true)
 |-- code_parametre_se: string (nullable = true)
 |-- code_prelevement: string (nullable = true)
 |-- code_type_parametre: string (nullable = true)
 |-- code_unite: string (nullable = true)
 |-- conclusion_conformite_prelevement: string (nullable = true)
 |-- conformite_limites_bact_prelevement: string (nullable = true)
 |-- conformite_limites_pc_prelevement: string (nullable = true)
 |-- conformite_references_bact_prelevement: string (nullable = true)
 |-- conformite_references_pc_prelevement: string (nullable = true)
 |-- date_prelevement: string (nullable = true)
 |-- libelle_parametre: string (nullable = true)
 |-- libelle_parametre_maj: string (nullable = true

26/05/04 16:20:46 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.



=== NOMBRE DE LIGNES: 280000 ===

=== PREMIÈRES LIGNES ===

=== STATISTIQUES NUMÉRIQUES ===


+-------+------------------+
|summary|resultat_numerique|
+-------+------------------+
|  count|            279370|
|   mean|13.147715933478894|
| stddev| 92.28910620346485|
|    min|             -34.0|
|    max|           30018.0|
+-------+------------------+



In [11]:
# 4. Analyse des valeurs nulles
print("\n=== VALEURS NULL PAR COLONNE ===")
for col in df_bronze.columns:
    null_count = df_bronze.filter(F.col(col).isNull()).count()
    print(f"{col}: {null_count} ({round(null_count/df_bronze.count()*100, 2)}%)")


=== VALEURS NULL PAR COLONNE ===
code_commune: 0 (0.0%)
code_departement: 0 (0.0%)


code_installation_amont: 176518 (63.04%)
code_lieu_analyse: 0 (0.0%)
code_parametre: 93 (0.03%)
code_parametre_cas: 116145 (41.48%)
code_parametre_se: 0 (0.0%)
code_prelevement: 0 (0.0%)
code_type_parametre: 0 (0.0%)
code_unite: 0 (0.0%)
conclusion_conformite_prelevement: 2 (0.0%)
conformite_limites_bact_prelevement: 2 (0.0%)
conformite_limites_pc_prelevement: 2 (0.0%)
conformite_references_bact_prelevement: 2 (0.0%)
conformite_references_pc_prelevement: 2 (0.0%)
date_prelevement: 0 (0.0%)
libelle_parametre: 0 (0.0%)
libelle_parametre_maj: 0 (0.0%)
libelle_parametre_web: 280000 (100.0%)
libelle_unite: 0 (0.0%)
limite_qualite_parametre: 133168 (47.56%)
nom_commune: 0 (0.0%)
nom_departement: 0 (0.0%)
nom_distributeur: 0 (0.0%)
nom_installation_amont: 176518 (63.04%)
nom_moa: 0 (0.0%)
nom_uge: 0 (0.0%)
reference_analyse: 30074 (10.74%)
reference_qualite_parametre: 219922 (78.54%)


reseaux: 0 (0.0%)
resultat_alphanumerique: 14 (0.01%)
resultat_numerique: 630 (0.22%)


In [13]:
# 5. Distribution des départements
print("\n=== RÉPARTITION PAR DÉPARTEMENT ===")
df_bronze.groupBy("nom_departement").count().orderBy(F.desc("count")).show()


=== RÉPARTITION PAR DÉPARTEMENT ===
+------------------+-----+
|   nom_departement|count|
+------------------+-----+
|    Seine-et-Marne|18918|
|            Savoie| 9240|
|             Isère| 7796|
|Meurthe-et-Moselle| 7734|
|              Oise| 6028|
|              Aube| 5291|
|     Côtes-d'Armor| 5288|
|          Morbihan| 5285|
|         Côte-d'Or| 5164|
|           Essonne| 5021|
|      Loir-et-Cher| 5000|
|          Calvados| 4860|
|            Sarthe| 4664|
|             Loire| 4602|
|     Pas-de-Calais| 4405|
|      Eure-et-Loir| 4231|
|    Seine-Maritime| 4152|
|            Vosges| 4116|
|              Nord| 4096|
|              Gard| 4058|
+------------------+-----+
only showing top 20 rows


In [14]:
# 6. Types de paramètres
print("\n=== DISTRIBUTION DES TYPES DE PARAMÈTRES ===")
df_bronze.groupBy("code_type_parametre").count().show()


=== DISTRIBUTION DES TYPES DE PARAMÈTRES ===
+-------------------+------+
|code_type_parametre| count|
+-------------------+------+
|                  O| 15148|
|                  N|264852|
+-------------------+------+



In [34]:

# 7. Analyse de conformité
print("\n=== STATUT DE CONFORMITÉ ===")
df_bronze.groupBy("conclusion_conformite_prelevement").count().show(truncate=False)




=== STATUT DE CONFORMITÉ ===
+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [33]:
# 8. Valeurs extrêmes pour les résultats numériques
print("\n=== MIN/MAX DES RÉSULTATS NUMÉRIQUES ===")
df_bronze.agg(
    F.min("resultat_numerique").alias("min"),
    F.max("resultat_numerique").alias("max"),
    F.avg("resultat_numerique").alias("avg")
).show()


=== MIN/MAX DES RÉSULTATS NUMÉRIQUES ===
+-----+-------+------------------+
|  min|    max|               avg|
+-----+-------+------------------+
|-34.0|30018.0|13.147715933478894|
+-----+-------+------------------+

